# LightGBM：大规模梯度提升的利器
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：LightGBM.py | 核心功能：直方图算法、Leaf-wise 生长、类别特征支持

## 概述

LightGBM 是微软开源的梯度提升框架。两大创新：GOSS（基于梯度的单边采样）和 EFB（互斥特征捆绑），训练速度比 XGBoost 快数倍。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, r2_score

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
# --- 输出结果 ---
    print("[SKIP] LightGBM 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_LGB = False
    print("LightGBM 未安装,请运行: pip install lightgbm\n")


## 数学原理

### 1. LightGBM 的核心创新

LightGBM（Light Gradient Boosting Machine）由微软提出，核心改进是**基于直方图的决策树算法**和**叶子优先生长策略**，大幅提升训练速度。

### 2. 直方图算法（Histogram-based Splitting）

传统 XGBoost 需要遍历所有特征值寻找最优分裂点，LightGBM 将连续特征离散化为 $k$ 个桶（bin）：

1. 对每个特征，将 $n$ 个样本值映射到 $k$ 个等频直方图桶中
2. 遍历 $k$ 个桶的分裂点（而非 $n$ 个值），时间复杂度从 $O(n \cdot d)$ 降至 $O(k \cdot d)$

$$\text{split cost} = O(k \cdot d) \ll O(n \cdot d), \quad k \ll n$$

### 3. 叶子优先生长（Leaf-wise Growth）

LightGBM 采用**叶子优先**策略（而非层序生长）：

$$j^* = \arg\max_{j \in \text{leaves}} \text{Gain}(j)$$

每步选择**增益最大的叶子节点**进行分裂，直到达到 `num_leaves` 限制。

与 XGBoost 的层序生长相比：
- 同样的 `num_leaves` 下，叶子优先能构建更深、更不对称的树
- 损失下降更快，但需要注意过拟合

### 4. Gradient-based One-Side Sampling (GOSS)

LightGBM 对样本进行智能采样：
- 保留梯度绝对值大的样本（信息量大）
- 随机丢弃梯度小的样本（已学好的样本）

设 $a$ 为大梯度样本比例，$b$ 为小梯度样本比例：

$$\tilde{g}_j = \frac{1}{n}\left(\sum_{x_i \in A_l} g_i + \frac{1-a}{b}\sum_{x_i \in B_l} g_i\right)$$

其中 $A_l$ 是大梯度集合，$B_l$ 是小梯度的随机子集，$\frac{1-a}{b}$ 是补偿系数。

### 5. Exclusive Feature Bundling (EFB)

高维稀疏数据中，许多特征是互斥的（不同时取非零值）。LightGBM 将互斥特征捆绑为一个特征，降低有效特征数：

$$\text{bundles} \ll \text{features}$$

这在高维稀疏数据中效果显著。

### 6. num_leaves 与 max_depth 的关系

LightGBM 的核心复杂度控制参数是 `num_leaves`（叶子数），而非 `max_depth`：

$$\text{num\_leaves} \leq 2^{\text{max\_depth}}$$

- `num_leaves=31`：最多 31 个叶节点，模型复杂度适中
- `max_depth` 作为辅助约束，`-1` 表示不限制深度

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `num_leaves=31` | 叶子优先生长直到 31 个叶子 |
| `max_depth=3` | 深度约束（辅助限制） |
| `learning_rate=0.1` | 步长收缩 $\nu$ |
| `subsample=0.8` | 行采样比例 |
| `colsample_bytree=0.8` | 列采样比例（随机子空间） |
| `min_data_in_leaf` | 叶节点最小样本数，防过拟合 |
| `early_stopping(50)` | 验证集 50 轮无提升则停止 |
| `best_iteration_` | 最优迭代次数 |

### 1. 分类任务

在分类任务上训练模型并评估性能。

In [ ]:
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 2. 基础 LightGBM

运行 2. 基础 LightGBM 的代码，观察算法在该环节的行为。

In [ ]:
lgb_clf = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
# --- 赋值/配置 ---
    colsample_bytree=0.8,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
# --- 训练模型 ---
lgb_clf.fit(X_train, y_train, eval_set=[(X_test, y_test)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

print("=== LightGBM 分类 ===")
print(f"训练准确率: {lgb_clf.score(X_train, y_train):.4f}")
print(f"测试准确率: {lgb_clf.score(X_test, y_test):.4f}")
print(f"最优轮数: {lgb_clf.best_iteration_}")

### 3. 特征重要性

分析各特征对模型预测的贡献度。

In [ ]:
print("\n=== 特征重要性 (gain) ===")
importances = lgb_clf.feature_importances_
for i in np.argsort(importances)[::-1]:
    print(f"  特征{i}: {importances[i]:.4f}")

### 4. 关键参数

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== 关键参数 ===")
print("num_leaves: 叶子数（核心参数）,比 max_depth 控制更细")
print("max_depth: 树的最大深度（-1=不限）")
print("min_data_in_leaf: 叶节点最小样本数,防过拟合")

### 5. num_leaves vs max_depth

运行 5. num_leaves vs max_depth 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== num_leaves 对比 ===")
for nl in [7, 15, 31, 63, 127]:
    m = lgb.LGBMClassifier(n_estimators=100, num_leaves=nl, random_state=42, verbose=-1)
    m.fit(X_train, y_train)
    print(f"  num_leaves={nl:>3}: 测试准确率={m.score(X_test, y_test):.4f}")

### 6. 回归任务

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== LightGBM 回归 ===")
X_r, y_r = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
lgb_reg = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
lgb_reg.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
# --- 继续 ---
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
print(f"R²: {lgb_reg.score(X_te, y_te):.4f}")

### 7. LightGBM vs XGBoost

运行 7. LightGBM vs XGBoost 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== LightGBM vs XGBoost ===")
print("Leaf-wise 生长: LightGBM 每次选增益最大的叶子分裂（非层级生长）")
print("直方图加速: 将连续值分桶,减少分裂点搜索时间")
print("单边梯度采样 (GOSS): 保留大梯度样本,随机采样小梯度样本")
print("互斥特征捆绑 (EFB): 将互斥稀疏特征合并")

print("\n=== LightGBM 要点 ===")
print("- num_leaves 是最核心的参数（控制模型复杂度）")
print("- 比 XGBoost 快 2~10 倍（大数据集优势明显）")
print("- 内存占用更低（直方图算法）")
print("- 更适合大规模数据和高维特征")
# --- 输出结果 ---
print("- 缺点：小数据集可能不如 XGBoost 稳定")

## 关键代码解释

```python
import lightgbm as lgb
model = lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.1)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(10)])
```

`num_leaves` 控制树的复杂度（LightGBM 用 Leaf-wise 生长，不同于 XGBoost 的 Level-wise）。

## 注意事项

1. **num_leaves** 是最重要的参数，通常 < 2^max_depth
2. **小数据集容易过拟合**：需要限制 min_data_in_leaf
3. **原生支持分类特征**：不需要 one-hot 编码

## 总结与延伸

以上代码展示了 **LightGBM** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **GOSS**：优先保留梯度大的样本
- **EFB**：将互斥特征捆绑，减少特征数
- **Dask-LightGBM**：分布式训练
